# DP2 Survey Property & Monster Bright-Star Spatial Masking with SkyKatana

**Environment:** Rubin Science Platform (`LSST` kernel, 32 GiB RAM Nublado container) or local `lsdb_env-py313`  
**References:**
- DP2 Survey Property Maps (`sp_maps`): https://dp2.lsst.io/products/maps/sp_maps.html
- DP2 Tutorial 204.2 (`the_monster` reference catalog): https://dp2.lsst.io/tutorials/notebook/204/notebook-204-2.html
- SkyKatana documentation: https://skykatana.readthedocs.io/en/latest/

### First-time setup on the RSP
Open an RSP terminal once and run:
```bash
./scripts/setup_rsp.sh
```
Then restart the notebook kernel (`LSST`) and execute the cells below.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import healsparse as hsp
from skykatana import SkyMaskPipe

from dp2_skykatana_mask import (
    DP2MaskBuilder,
    load_config,
)
from dp2_skykatana_mask.memory import MemoryGuard, configure_rsp_thread_env

configure_rsp_thread_env(max_threads=2)
guard = MemoryGuard(max_ram_gib=32.0)
print('Container memory check:', guard.snapshot().format_short())

## 1. Connect to DP2 Butler and Inspect `sp_maps` & `the_monster`

On the RSP (`data.lsst.cloud`), `Butler("dp2", collections=["dp2"])` exposes:
- 14 consolidated `HealSparseMap` survey property maps (`nside_coverage=32`, `nside_sparse=32768`, `skymap='lsst_cells_v2'`), including:
  - `deepCoadd_exposure_time_consolidated_map_sum` (accumulated seconds; divide by `30 s` for standard visit equivalents)
  - `deepCoadd_psf_maglim_consolidated_map_weighted_mean` ($5\sigma$ point-source AB depth)
  - `deepCoadd_psf_size_consolidated_map_weighted_mean` (PSF determinant radius in $0.2''$ pixels)
- `the_monster_20250219` (`htm7`-sharded `SimpleCatalog` with `coord_ra`/`coord_dec` in radians and `monster_ComCam_{u..y}_flux` in `nJy`).

If you open this notebook locally outside the RSP without an access token, it automatically falls back to `SyntheticDP2Butler` so every cell remains runnable.

In [ ]:
from dp2_skykatana_mask.butler_io import (
    discover_monster_dataset,
    list_consolidated_maps,
    open_dp2_butler,
)
from dp2_skykatana_mask.synthetic import SyntheticDP2Butler

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
pilot_cfg = load_config(REPO_ROOT / 'configs' / 'dp2_ecdfs_pilot.yaml')

try:
    butler = open_dp2_butler(pilot_cfg)
    USING_SYNTHETIC = False
    print('Connected to live RSP DP2 Butler!')
except ImportError:
    butler = SyntheticDP2Butler(
        center_ra_deg=pilot_cfg.region.ra_deg,
        center_dec_deg=pilot_cfg.region.dec_deg,
        radius_deg=pilot_cfg.region.radius_deg,
        nside_sparse=1 << 13,
        n_stars=250,
    )
    pilot_cfg.property_maps.order_sparse = 13
    pilot_cfg.monster_stars.order_sparse = 13
    pilot_cfg.output.order_out = 13
    USING_SYNTHETIC = True
    print('lsst.daf.butler not found locally -> using SyntheticDP2Butler demo.')

sp_maps = list_consolidated_maps(butler)
monster_ds = discover_monster_dataset(butler)
print(f'Found {len(sp_maps)} consolidated sp_maps and Monster catalog: {monster_ds}')
for ds in sp_maps:
    print('  -', ds)

## 2. Pilot Run on ECDFS (`dp2_ecdfs_pilot.yaml`)

Before building the full 6,000+ deg$^2$ DP2 mask, validate your per-band exposure, `min_filters`, `min_maglim`, and Monster bright-star radius parameters on a $1.5^\circ$ cone around ECDFS (~30-60 seconds on RSP).

In [ ]:
pilot_cfg.output.output_dir = str(REPO_ROOT / 'outputs' / 'notebook_ecdfs_pilot')
builder = DP2MaskBuilder(pilot_cfg, butler=butler)
print('Analytic RAM budget (GiB):', json.dumps(builder.estimate_memory(assumed_area_deg2=8.0), indent=2))

pipe, summary = builder.run()
print(json.dumps(summary.to_dict(), indent=2))

In [ ]:
# Inspect SkyKatana stages and display the generated QA figure
print('SkyMaskPipe stages populated:')
for stage_name in ('propmask', 'starmask', 'boxmask', 'mwmask', 'mask'):
    st = getattr(pipe, stage_name, None)
    if st is not None:
        print(f'  {stage_name:10s}: nside={st.nside_sparse:6d}, valid_area={st.get_valid_area(degrees=True):8.3f} deg^2, bit_packed={st.is_bit_packed_map}')

qa_img = plt.imread(summary.output_files['qa_plot_png'])
fig, ax = plt.subplots(figsize=(13, 5))
ax.imshow(qa_img)
ax.axis('off')
plt.show()

## 3. Applying the Mask to Catalogs (`SkyMaskPipe.apply`) & HATS / LSDB

Once built (or reloaded from disk via `SkyMaskPipe.read(...)` or `healsparse.HealSparseMap.read(...)`), you can filter DP2 object tables, shear catalogs, or `lsdb` HATS partitions in seconds.

In [ ]:
# Reload the persisted SkyMaskPipe directory to verify disk round-trip
reloaded_pipe = SkyMaskPipe.read(summary.output_files['skykatana_pipe_dir'], verbose=False)

# Sample test catalog across the field and filter with SkyMaskPipe.apply
rng = np.random.default_rng(2026)
test_cat = pd.DataFrame({
    'ra': rng.uniform(51.8, 54.4, size=10_000),
    'dec': rng.uniform(-29.3, -26.8, size=10_000),
})
clean_cat = reloaded_pipe.apply(stage='mask', cat=test_cat, columns=('ra', 'dec'))
print(f'Input sources: {len(test_cat):,} -> Inside clean DP2 footprint & outside Monster star halos: {len(clean_cat):,}')

## 4. Scaling to the Full DP2 Footprint on 32 GiB RSP Pods

For the full DP2 survey footprint (`configs/dp2_wfd_default.yaml` or `configs/dp2_deep_uniform.yaml`), run either from an RSP terminal (recommended so browser disconnects cannot interrupt long runs):

```bash
tmux new -s dp2mask
./scripts/run_rsp_mask.sh configs/dp2_wfd_default.yaml
```

or directly in Python below by flipping `RUN_FULL_DP2 = True`.

In [ ]:
RUN_FULL_DP2 = False

if RUN_FULL_DP2 and not USING_SYNTHETIC:
    full_cfg = load_config(REPO_ROOT / 'configs' / 'dp2_wfd_default.yaml')
    full_builder = DP2MaskBuilder(full_cfg, butler=butler)
    print('Full DP2 memory estimate (GiB):', full_builder.estimate_memory(assumed_area_deg2=6500.0))
    full_pipe, full_summary = full_builder.run()
    print(json.dumps(full_summary.to_dict(), indent=2))

## Summary & Output Products

| Product | Filename | Downstream Usage |
| :--- | :--- | :--- |
| Native SkyKatana pipeline | `skykatana_pipe/` | `SkyMaskPipe.read(...)` -> `.apply()`, `.makerans()`, stage inspection |
| Standalone bit-packed HealSparse | `dp2_combined_mask.hs` | `healsparse.HealSparseMap.read(...)` / LSST & DESC pipelines |
| IVOA Multi-Order Coverage | `dp2_combined_mask_moc.fits` | `lsdb.read_hats(..., search_filter=MOCSearch(...))`, `ipyaladin`, TOPCAT |
| Fractional detection map | `dp2_fracdet_o10.hs` | Angular clustering & pixel weight corrections |
| Uniform random catalog | `dp2_randoms.parquet` | Two-point correlation functions ($w(\theta)$, $C_\ell$) |
| Build provenance & RAM log | `mask_build_summary.json` | Exact area waterfall, star counts, and peak pod RSS |